In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize,sent_tokenize
import re,string,unicodedata
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score


In [51]:
df = pd.read_csv("IMDB Dataset.csv")

In [52]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [53]:
df.shape

(50000, 2)

In [54]:
df['sentiment'].value_counts()

positive    25000
negative    25000
Name: sentiment, dtype: int64

In [55]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [57]:
df['sentiment'].replace({'positive':1 , "negative":0},inplace=True)

In [58]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


## TEXT PRE-PROCESSING

###  FUNCTION FOR REMOVAL OF HTML TAGS

In [59]:
import re
def remove_html_tags(text):
    pattern = r"<.*?>"
    text = re.sub(pattern, "", text)
    return text


In [60]:
df["review"] = df["review"].apply(remove_html_tags)
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wo

### FUNCTION TO LOWER THE TEXT

In [1]:
def convert_lower(text):
    return text.lower()

In [62]:
df['review']=df['review'].apply(convert_lower)
df['review'][0]

### FUNCTION TO REMOVE SPECIAL CHARACTERS

In [1]:
def remove_special_char(text):
    x=''
    
    for i in text:
        if i.isalnum():
            x = x + i
        else:
            x = x + ' '
    return x

In [2]:
df['review']=df['review'].apply(remove_special_char)
df['review'][0]

NameError: name 'df' is not defined

### FUNCTION TO REMOVE STOPWORDS

In [14]:
pip install nltk

                                              0.0/1.5 MB ? eta -:--:--
     -----------------                        0.7/1.5 MB 10.2 MB/s eta 0:00:01
     -------------------------                1.0/1.5 MB 7.5 MB/s eta 0:00:01
     -----------------------------            1.1/1.5 MB 6.3 MB/s eta 0:00:01
     --------------------------------------   1.4/1.5 MB 5.7 MB/s eta 0:00:01
     ---------------------------------------- 1.5/1.5 MB 5.1 MB/s eta 0:00:00
                                              0.0/268.1 kB ? eta -:--:--
     ------------------------------------- 268.1/268.1 kB 16.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [15]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stopwordss = stopwords.words('english')

[nltk_data] Downloading package stopwords to C:\Users\Sanika
[nltk_data]     Rane\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [75]:
stopwordss

['i',
 'me',
 'my',
 'myself',
 'we',
 'our',
 'ours',
 'ourselves',
 'you',
 "you're",
 "you've",
 "you'll",
 "you'd",
 'your',
 'yours',
 'yourself',
 'yourselves',
 'he',
 'him',
 'his',
 'himself',
 'she',
 "she's",
 'her',
 'hers',
 'herself',
 'it',
 "it's",
 'its',
 'itself',
 'they',
 'them',
 'their',
 'theirs',
 'themselves',
 'what',
 'which',
 'who',
 'whom',
 'this',
 'that',
 "that'll",
 'these',
 'those',
 'am',
 'is',
 'are',
 'was',
 'were',
 'be',
 'been',
 'being',
 'have',
 'has',
 'had',
 'having',
 'do',
 'does',
 'did',
 'doing',
 'a',
 'an',
 'the',
 'and',
 'but',
 'if',
 'or',
 'because',
 'as',
 'until',
 'while',
 'of',
 'at',
 'by',
 'for',
 'with',
 'about',
 'against',
 'between',
 'into',
 'through',
 'during',
 'before',
 'after',
 'above',
 'below',
 'to',
 'from',
 'up',
 'down',
 'in',
 'out',
 'on',
 'off',
 'over',
 'under',
 'again',
 'further',
 'then',
 'once',
 'here',
 'there',
 'when',
 'where',
 'why',
 'how',
 'all',
 'any',
 'both',
 'each

In [76]:
def remove_stopwords(text):
    x=[]
    for i in text.split():
        
        if i not in stopwords.words('english'):
            x.append(i)
    y=x[:]
    x.clear()
    return y
    

In [77]:
# df['review']=df['review'].apply(remove_stopwords)

In [78]:
# df['review'][0]

['one',
 'reviewers',
 'mentioned',
 'watching',
 '1',
 'oz',
 'episode',
 'hooked',
 'right',
 'exactly',
 'happened',
 'first',
 'thing',
 'struck',
 'oz',
 'brutality',
 'unflinching',
 'scenes',
 'violence',
 'set',
 'right',
 'word',
 'go',
 'trust',
 'show',
 'faint',
 'hearted',
 'timid',
 'show',
 'pulls',
 'punches',
 'regards',
 'drugs',
 'sex',
 'violence',
 'hardcore',
 'classic',
 'use',
 'word',
 'called',
 'oz',
 'nickname',
 'given',
 'oswald',
 'maximum',
 'security',
 'state',
 'penitentary',
 'focuses',
 'mainly',
 'emerald',
 'city',
 'experimental',
 'section',
 'prison',
 'cells',
 'glass',
 'fronts',
 'face',
 'inwards',
 'privacy',
 'high',
 'agenda',
 'em',
 'city',
 'home',
 'many',
 'aryans',
 'muslims',
 'gangstas',
 'latinos',
 'christians',
 'italians',
 'irish',
 'scuffles',
 'death',
 'stares',
 'dodgy',
 'dealings',
 'shady',
 'agreements',
 'never',
 'far',
 'away',
 'would',
 'say',
 'main',
 'appeal',
 'show',
 'due',
 'fact',
 'goes',
 'shows',
 'da

In [81]:
# df.to_csv('stopword_removed_csv.csv')

### FUNCTION FOR STEMMING

In [82]:
# Performing stemming
ps=PorterStemmer()

y=[]
def stem_words(text):
    for i in text:
        y.append(ps.stem(i))
    z=y[:]
    y.clear()
    return z

In [83]:
df['review']=df['review'].apply(stem_words)
df['review'][0]

['one',
 'review',
 'mention',
 'watch',
 '1',
 'oz',
 'episod',
 'hook',
 'right',
 'exactli',
 'happen',
 'first',
 'thing',
 'struck',
 'oz',
 'brutal',
 'unflinch',
 'scene',
 'violenc',
 'set',
 'right',
 'word',
 'go',
 'trust',
 'show',
 'faint',
 'heart',
 'timid',
 'show',
 'pull',
 'punch',
 'regard',
 'drug',
 'sex',
 'violenc',
 'hardcor',
 'classic',
 'use',
 'word',
 'call',
 'oz',
 'nicknam',
 'given',
 'oswald',
 'maximum',
 'secur',
 'state',
 'penitentari',
 'focus',
 'mainli',
 'emerald',
 'citi',
 'experiment',
 'section',
 'prison',
 'cell',
 'glass',
 'front',
 'face',
 'inward',
 'privaci',
 'high',
 'agenda',
 'em',
 'citi',
 'home',
 'mani',
 'aryan',
 'muslim',
 'gangsta',
 'latino',
 'christian',
 'italian',
 'irish',
 'scuffl',
 'death',
 'stare',
 'dodgi',
 'deal',
 'shadi',
 'agreement',
 'never',
 'far',
 'away',
 'would',
 'say',
 'main',
 'appeal',
 'show',
 'due',
 'fact',
 'goe',
 'show',
 'dare',
 'forget',
 'pretti',
 'pictur',
 'paint',
 'mainstrea

### FUNCTION FOR LEMETIZATION

In [ ]:
# wnl = nltk.WordNetLemmatizer()

# lemmatized_words = []
# def lemmatize_words(text):
#     for word in text:
#         lemmatized_words.append(wnl.lemmatize(word))
        
#     z=lemmatized_words[:]
#     lemmatized_words.clear()
    return z

In [ ]:
# df['review']=df['review'].apply(stem_words)
# df['review'][0]

### FUNCTION TO CONCATENATE THE ORIGNAL REVIEW

In [84]:
def join_back(list_input):
    return ' '.join(list_input)

In [85]:
df['review']=df['review'].apply(join_back)
df['review'][0]

'one review mention watch 1 oz episod hook right exactli happen first thing struck oz brutal unflinch scene violenc set right word go trust show faint heart timid show pull punch regard drug sex violenc hardcor classic use word call oz nicknam given oswald maximum secur state penitentari focus mainli emerald citi experiment section prison cell glass front face inward privaci high agenda em citi home mani aryan muslim gangsta latino christian italian irish scuffl death stare dodgi deal shadi agreement never far away would say main appeal show due fact goe show dare forget pretti pictur paint mainstream audienc forget charm forget romanc oz mess around first episod ever saw struck nasti surreal say readi watch develop tast oz got accustom high level graphic violenc violenc injustic crook guard sold nickel inmat kill order get away well manner middl class inmat turn prison bitch due lack street skill prison experi watch oz may becom comfort uncomfort view that get touch darker side'

In [86]:
# df.to_csv('stopword_removed.csv')

In [3]:
df=pd.read_csv('stopword_removed.csv')
df.drop(columns=['Unnamed: 0'],inplace=True)
df.head(3)

NameError: name 'pd' is not defined

### COUNTVECTORIZER

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000)
X=cv.fit_transform(df['review']).toarray()
print(X)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [6]:
print(X.shape)
X[0].max()

(50000, 5000)


6

In [4]:
y=df.iloc[:,-1].values
print(y.shape)
y

NameError: name 'df' is not defined

In [8]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [9]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=10)

print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(40000, 5000) (10000, 5000) (40000,) (10000,)


In [6]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.metrics import accuracy_score

In [94]:
clf1=GaussianNB()

clf1.fit(X_train,y_train)

y_pred1 = clf1.predict(X_test)

print('Gaussian', accuracy_score(y_test, y_pred1))

Gaussian 0.7222


In [8]:
clf2=MultinomialNB()

clf2.fit(X_train,y_train)

y_pred2 = clf2.predict(X_test)

print('MultinomialNB', accuracy_score(y_test, y_pred2))

NameError: name 'X_train' is not defined

In [12]:
clf3=BernoulliNB()

clf3.fit(X_train,y_train)

y_pred3 = clf3.predict(X_test)

accuracy=accuracy_score(y_test, y_pred3)

print('BernoulliNB', round(accuracy,2))
print('BernoulliNB Confusion Matrix',confusion_matrix(y_test, y_pred3))




BernoulliNB 0.85
BernoulliNB Confusion Matrix [[4347  761]
 [ 710 4182]]


In [1]:
import pickle
pickle.dump(cv,open('count-vectorizer.pkl' , "wb"))
pickle.dump(clf3,open('movie_review_classification.pkl',"wb"))

NameError: name 'cv' is not defined

In [3]:
import pickle
save_cv = pickle.load(open('count-vectorizer.pkl' , 'rb'))
model= pickle.load(open('movie_review_classification.pkl' , 'rb'))

In [4]:
def test_model(sentence):
    sen = save_cv.transform([sentence]).toarray()
    res = model.predict(sen)[0]
    if res == 1:
        return "Positive Review"
    else:
        return 'Negative review'

In [5]:
sen= ''' 


For prabhas only giving star ratings!!
He is super as always no character should be blamed here but picture was not up to 
the mark it didn't give any ramayana feels very much disappointed, everything looks soo animated like avatar  
holywood movie ravana's Lanka!!! Where was it?? And his look oh god!!! Also sita is not opt for the character 
i guess they should have taken who gives the vibe and with round face that's my openion no offence here kriti 
fans so over all it wasn't what we expected just for prabhas only movie got such hype and running successfully it 
looks like director created with his own imagination giving hollywood bollywood touch which was not needed they should
have choosed actors from south india only anyway except prabhas there is nothing interested in the movie also should 
mention the songs are very good actually it didn't give that beautiful vibe with video as it gives just in the audio. 
And sorry to say it but this movie is no where to the bahubali i am not comparing there is nothing to compare it would be s
illy to compare adipurush with BAHUBALI the setup was soo royal and realistic and in adipurush no setup which gives the 
palace and lanka vibe. Hindi sita k ram was good actually and good Sita and ravana were in that sorry no offence pleasee🙏✋✋✋✋

😶😶😶

'''
res = test_model(sen)
print(res)

Negative review


In [6]:
sen= ''' 

The film's all India total now stands at around Rs 277 cr– a huge number which falls
drastically short in comparison to its production budget, which is reported to be around Rs 500 cr. 
Adipurush is likely to wrap its Hindi lifetime collection with under Rs 150 cr, emerging as a flop at the box office.27

'''
res = test_model(sen)
print(res)

Negative review


In [ ]:
import pickle
import streamlit as st
import tkinter as tk
from tkinter import filedialog
import threading

# Load the sentiment analysis model
with open('movie_review_classification.pkl', 'rb') as f:
    model = pickle.load(f)

# Load the CountVectorizer object
with open('count-vectorizer.pkl', 'rb') as f:
    save_cv = pickle.load(f)

# Create a function to test the sentiment analysis model
def test_model(sentence):
    sen = save_cv.transform([sentence]).toarray()
    res = model.predict(sen)[0]
    if res == 1:
        return "Positive Review"
    else:
        return 'Negative review'

# Create a function to count the number of positive and negative reviews
def count_reviews(pdf_file):
    with open(pdf_file, 'rb') as f:
        content = f.read()

    # Split the content into sentences
    sentences = content.splitlines()

    # Initialize the counters
    positive_sentiments = 0
    negative_sentiments = 0

    # Iterate over the sentences
    for sentence in sentences:
        # Test the sentiment of the sentence
        sentiment = test_model(sentence)

        # Count the number of positive and negative sentiments
        if sentiment == 'Positive Review':
            positive_sentiments += 1
        elif sentiment == 'Negative review':
            negative_sentiments += 1

    return positive_sentiments, negative_sentiments

# Create a thread to run the file dialog
def open_file():
    global pdf_file
    pdf_file = filedialog.askopenfilename()

# Start the thread
thread = threading.Thread(target=open_file)
thread.start()

# Wait for the thread to finish
thread.join()

# If a PDF file was uploaded, analyze it
if pdf_file:
    prediction, positive_sentiments, negative_sentiments = count_reviews(pdf_file)

    st.write(f'The sentiment of the PDF file is {prediction}')
    st.write(f'The number of positive sentiments is {positive_sentiments}')
    st.write(f'The number of negative sentiments is {negative_sentiments}')

In [1]:
import streamlit as st
import os

def run_py_file(file_path):
    with open(file_path, 'r') as f:
        code = f.read()

    st.code(code)
    st.execute_code(code)

if __name__ == '__main__':
    file_path = st.file_uploader('Upload a PDF file')
    if file_path:
        run_py_file(file_path)

2023-07-23 15:57:18.261 
  command:

    streamlit run C:\Users\Sanika Rane\.conda\envs\streamlit\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
